This is a seq2seq model, which uses Keras's Sequential Model.

The model contains two Bidirectional LSTM layers, an Embedding layer, and dropout.

This notebook is work in progress.

In [30]:
import os, collections

import pandas as pd
import numpy as np

from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split

from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding, Bidirectional, RepeatVector, Dropout, TimeDistributed
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

from tensorflow.keras.losses import sparse_categorical_crossentropy

In [2]:
from google.colab import drive

drive.mount('/content/gdrive')
root_path = 'gdrive/My Drive/your_project_folder/' 

Mounted at /content/gdrive


In [3]:
def read_data_from_file(filename, data_dict):

    with open(filename) as fp:
        line = fp.readline()
        while line:
            bo, ch, ve, text = tuple(line.strip().split('\t'))
            words = text.split()
            for w in words:  
                # in the output data, composite placenames have a '_', which cannot be found in the input data
                words_split = w.split('_')               
                for word_split in words_split:
                    data_dict[bo].append(word_split)
        
            line = fp.readline()
            
    return data_dict

In [4]:
input_file = '/content/gdrive/My Drive/data/t-in_voc'
#input_file = 't-in_voc'
input_data = collections.defaultdict(list)

output_file = '/content/gdrive/My Drive/data/t-out'
#output_file = 't-out'
output_data = collections.defaultdict(list)

input_data = read_data_from_file(input_file, input_data)
output_data = read_data_from_file(output_file, output_data)

In [5]:
print(len(input_data['Gen']))
print(len(output_data['Gen']))

20611
20611


In [6]:
input_data['Gen'][0:10]

['B.:R;>CIJT',
 'B.@R@>',
 '>:ELOHIJM',
 '>;T',
 'HAC.@MAJIM',
 'W:>;T',
 'H@>@REY',
 'W:H@>@REY',
 'H@J:T@H',
 'TOHW.']

In [7]:
output_data['Gen'][0:10]

['B-R>CJT/',
 'BR>[',
 '>LH(J(M/JM',
 '>T',
 'H-CMJ(M/(JM',
 'W->T',
 'H->RY/:a',
 'W-H->RY/:a',
 'HJ(H[&TH',
 'THW/']

In [8]:
def make_in_out_sequences(data_dict, sequence_length):
    
    all_sequences = []
    for words_list in data_dict.values():

        for w in range(len(words_list) - sequence_length + 1):
    
            seq = ' '.join([words_list[ind] for ind in list(range(w, w + sequence_length))])
        
            # remove some special signs from output data (':', and '='). These only make the sequences longer.
            seq = seq.replace("=", "").replace(":a", "a").replace(":c", "c").replace(":d", "d").replace(":du", "du")
            all_sequences.append(seq)
        
    return all_sequences

In [9]:
sequence_length = 1

all_in_seqs = make_in_out_sequences(input_data, sequence_length)
all_out_seqs = make_in_out_sequences(output_data, sequence_length)


In [10]:
all_in_seqs[0:10]

['B.:R;>CIJT',
 'B.@R@>',
 '>:ELOHIJM',
 '>;T',
 'HAC.@MAJIM',
 'W:>;T',
 'H@>@REY',
 'W:H@>@REY',
 'H@J:T@H',
 'TOHW.']

In [11]:
print(len(all_in_seqs))
print(len(all_out_seqs))

300676
300676


In [12]:
for i in range(206000,206020):
  print(all_in_seqs[i], '---', all_out_seqs[i])

B.:BOW>@M --- B-!!BW>[/+M
J@BOW> --- !J!BW>[
W.B:Y;>T@M --- W-B-!!(JY>[/T+M
J;Y;>W. --- !J!(JY>[W
W.BAXAG.IJM --- W-B-(H-XG/JM
W.BAM.OW<:ADIJM --- W-B-(H-MW<D/JM
T.IH:JEH --- !T!HJH[
HAM.IN:X@H --- H-MNX(H/H
>;JP@H --- >JP(H/H
LAP.@R --- L-(H-PR/a
W:>;JP@H --- W->JP(H/H
L@>AJIL --- L-(H->JL/a
W:LAK.:B@FIJM --- W-L-(H-KBF/JM
MAT.AT --- MTT/
J@DOW --- JD/+W
W:CEMEN --- W-CMN/
HIJN --- HJN/
L@>;JP@H --- L-(H->JP(H/H
W:KIJ --- W-KJ
JA<:AFEH --- !J!<FH[


In [13]:
def prepare_train_data(input_data, output_data):

    input_seqs = []
    output_seqs = []
    input_chars = set()
    output_chars = set()

    # iterate over all the books
    for seq in range(len(input_data)): 
      
        #if len(output_data[seq]) > 40:
        #  continue
          
        if "*" in input_data[seq]: # cases of ketiv/qere are complicated, just skip them!
          continue

        input_list = list(input_data[seq])

        output_list = list(output_data[seq])
        #output_list = ['\t'] + output_list + ['\n']

        input_seqs.append(input_list)
        output_seqs.append(output_list)

        for input_ch in input_list:
            input_chars.add(input_ch)
        
        for output_ch in output_list:
            output_chars.add(output_ch)
                
    
    input_chars = sorted(list(input_chars))
    output_chars = sorted(list(output_chars))
    
    max_len_input = max([len(seq) for seq in input_seqs])
    max_len_output = max([len(seq) for seq in output_seqs])
    
    # shuffle the data. The model will get the data in small batches, it is preferable if the batches are more or less homogeneous
    # of course the inputs and outputs have to be shuffled identically
    input_seqs, output_seqs = shuffle(input_seqs, output_seqs)
    
    return input_seqs, output_seqs, input_chars, output_chars, max_len_input, max_len_output

In [14]:
def create_dicts(input_voc, output_voc):
    
    # these dicts map the input sequences
    input_idx2char = {}
    input_char2idx = {}
    
    input_idx2char[0] = 'PAD'
    input_char2idx['PAD'] = 0

    for k, v in enumerate(input_voc):
        input_idx2char[k + 1] = v
        input_char2idx[v] = k + 1
     
    # and these dicts map the output sequences of parts of speech
    output_idx2char = {}
    output_char2idx = {}
    
    output_idx2char[0] = 'PAD'
    output_char2idx['PAD'] = 0
    
    for k, v in enumerate(output_voc):
        output_idx2char[k + 1] = v
        output_char2idx[v] = k + 1
        
    return input_idx2char, input_char2idx, output_idx2char, output_char2idx

In [15]:
def pad(x, length=None):

    padded_sequences = pad_sequences(x, length, padding='post')
    
    return padded_sequences

In [99]:
def model_bidirectional(input_shape, output_sequence_length, input_vocab_size, output_vocab_size):

    model = Sequential()
    model.add(Embedding(input_vocab_size + 1, 12, input_shape=input_shape[1:]))
    
    model.add(Bidirectional(LSTM(512, return_sequences=False)))
    model.add(Dropout(0.25))

    model.add(RepeatVector(output_sequence_length))
    model.add(Bidirectional(LSTM(256, return_sequences=True)))
    
    model.add(Dropout(0.25))
    model.add(TimeDistributed(Dense(output_vocab_size + 1, activation='softmax')))


    
    adam = Adam(lr=0.0006, beta_1=0.995, beta_2=0.999, epsilon=0.00000001)
    model.compile(optimizer=adam, loss=sparse_categorical_crossentropy, metrics=['accuracy'])

    #learning_rate = 1e-3
    #model.compile(loss=sparse_categorical_crossentropy,
    #              optimizer=Adam,
    #              metrics=['accuracy'])
    return model

In [100]:
input_seqs, output_seqs, input_chars, output_chars, max_len_input, max_len_output = prepare_train_data(all_in_seqs, all_out_seqs)
print(len(input_seqs))

299488


In [101]:
input_seqs[0]

['J', '@', 'B', 'O', 'W', '>']

In [102]:
input_idx2char, input_char2idx, output_idx2char, output_char2idx = create_dicts(input_chars, output_chars)

In [103]:
input_seqs_num = []
output_seqs_num = []

for seq in input_seqs:
    input_seqs_num.append([input_char2idx[char] for char in seq])
    
for seq in output_seqs:
    output_seqs_num.append([output_char2idx[char] for char in seq])

In [104]:
input_seqs_num = pad(input_seqs_num)
output_seqs_num = pad(output_seqs_num)

# sparse_categorical_crossentropy function requires the labels to be in 3 dims
output_seqs_num = output_seqs_num.reshape(*output_seqs_num.shape, 1)

In [105]:
max_len_input = max([len(seq) for seq in input_seqs_num])
max_len_output = max([len(seq) for seq in output_seqs_num])
max_len_output

26

In [106]:
prepared_train_x = pad(input_seqs_num, max_len_output)

In [107]:
X_train, X_test, y_train, y_test = train_test_split(prepared_train_x, output_seqs_num, test_size=0.1, random_state=42)

In [109]:
rnn_model = model_bidirectional(
    X_train.shape,
    max_len_output,
    len(input_chars),
    len(output_chars))
    
rnn_model.summary()

callback = EarlyStopping(monitor='val_loss', patience=5, verbose=0, mode='auto')
rnn_model.fit(X_train, y_train, batch_size=256, epochs=30, validation_split=0.1, callbacks=[callback])


/usr/local/lib/python3.7/dist-packages/tensorflow/python/keras/optimizer_v2/optimizer_v2.py:375: UserWarning: The `lr` argument is deprecated, use `learning_rate` instead.
  "The `lr` argument is deprecated, use `learning_rate` instead.")


Model: "sequential_6"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
embedding_6 (Embedding)      (None, 26, 12)            396       
_________________________________________________________________
bidirectional_12 (Bidirectio (None, 1024)              2150400   
_________________________________________________________________
dropout_11 (Dropout)         (None, 1024)              0         
_________________________________________________________________
repeat_vector_6 (RepeatVecto (None, 26, 1024)          0         
_________________________________________________________________
bidirectional_13 (Bidirectio (None, 26, 512)           2623488   
_________________________________________________________________
dropout_12 (Dropout)         (None, 26, 512)           0         
_________________________________________________________________
time_distributed_6 (TimeDist (None, 26, 41)           

In [110]:
correct = 0

preds = rnn_model.predict(X_test)

for i in range(len(preds)):
  
    pred_output = [output_idx2char[np.argmax(pred)] for pred in preds[i] if output_idx2char[np.argmax(pred)] != 'PAD']
    
    true_output = [output_idx2char[char[0]] for char in y_test[i]  if output_idx2char[char[0]] != 'PAD']

    if pred_output == true_output:
      correct += 1

    print('predicted: ', ''.join(pred_output))
    print('true: ', ''.join(true_output))
    print(' ')

accuracy = correct / len(preds)

print(accuracy)

Streaminguitvoer ingekort tot de laatste 5000 regels.
 
predicted:  YDQJH/
true:  YDQJH/
 
predicted:  W->CPR/
true:  W->CPR/
 
predicted:  KJ
true:  KJ
 
predicted:  B->P/+J
true:  B->P/+J
 
predicted:  >CR
true:  >CR
 
predicted:  L+J
true:  L+J
 
predicted:  ]N]CB<[T
true:  ]N]CB<[T
 
predicted:  CM/+K
true:  CM/+K
 
predicted:  M(N-JRWCLM/
true:  M(N-JRWCLM/
 
predicted:  TRX/
true:  TRX/
 
predicted:  W:n-!J!](H](NGD[
true:  W:n-!J!](H](NGD[
 
predicted:  B-MYWD(H/T+J
true:  B-MYWD(H/T+J
 
predicted:  B-<JN/J+K
true:  B-<JN/J+K
 
predicted:  W-!!(X(([+HW
true:  W-!!(LQX[~N+(HW
 
predicted:  BN/J
true:  BN/J
 
predicted:  !J!BNH[
true:  !J!BNH[
 
predicted:  W:n-!J!](H](J&WLD[
true:  W:n-!J!](H](J&WLD[
 
predicted:  M(N-L-PN(H/J
true:  M(N-L-PN(H/J
 
predicted:  H-<Z/JM
true:  H-<Z/JM
 
predicted:  W-<D
true:  W-<D
 
predicted:  TWL<(T/T
true:  TWL<(T/T
 
predicted:  !J!HJ(H[
true:  !J!HJ(H[
 
predicted:  HN(H+NJ
true:  HN(H+NJ
 
predicted:  K-RYWN/+W
true:  K-RYWN/+W
 
predicted: 